# Day 2 — Basic Model Building (and the things to know around it)

Goal: go from *"I ran a model once"* to *"I can build one properly and judge it honestly"*.

1. The ML workflow (recap)
2. Loading real data (CSV)
3. Features, target, and the train/test split
4. Preprocessing: missing values, encoding, scaling
5. Building models: baseline, logistic regression, KNN, decision tree, random forest
6. Evaluating honestly: accuracy and its traps, precision/recall/F1, regression metrics
7. Overfitting and underfitting
8. Cross-validation
9. Pipelines
10. Saving a trained model

**Run the cells in order, top to bottom.** Later cells depend on earlier ones.

## 1. Recap: the ML workflow

Day 1 ended with this loop:

**load data → split train/test → fit model → predict → evaluate**

Yesterday the data was small and clean (the iris flowers). Today we repeat the loop on **messier, more realistic data** — with missing values and text columns — and we slow down on the two steps beginners usually skip: **preparing the data** and **judging the result honestly**. That judgment is what separates people who merely run models from people who can build them.

In [ ]:
# Everything we need today, in one place
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('NumPy', np.__version__, '| pandas', pd.__version__)

## 2. Loading real data with `pd.read_csv`

Real projects start from a file — usually a **CSV** (a plain-text table) downloaded from Kaggle or exported from somewhere. To practice that exact flow *inside this notebook* (no downloads needed), we will:

1. **generate** a small but realistic table — a customer-churn dataset (`churned` = did the customer cancel their subscription?),
2. **save** it as `customers.csv`,
3. **read it back** with `pd.read_csv` — exactly what you will do with a Kaggle dataset.

The pattern hidden in the data: customers churn more when they are on **month-to-month contracts**, pay **higher charges**, call **support often**, and have been with the company only a **short time**. The model's job today is to *rediscover* that pattern from just 200 rows.

In [ ]:
rng = np.random.default_rng(42)          # seed 42 -> same 'random' data every run
n = 200                                  # number of customers

# 1. the raw ingredients
contract        = rng.choice(['month-to-month', 'one-year', 'two-year'], size=n, p=[0.55, 0.25, 0.20])
tenure_months   = rng.integers(1, 60, size=n)          # how long a customer has been with the company
monthly_charges = rng.normal(60, 18, size=n).round(2)  # amount paid per month
support_calls   = rng.poisson(2.2, size=n)             # calls to support in the last 3 months

# 2. the hidden rule that decides churn (the model must NOT see this formula!)
score   = 0.9*(contract == 'month-to-month') - 0.04*tenure_months + 0.03*(monthly_charges - 60) + 0.45*support_calls - 1.2
p_churn = 1/(1 + np.exp(-score))                       # turn the score into a probability between 0 and 1
churned = rng.binomial(1, p_churn)                     # flip a weighted coin for each customer

# 3. build the table, save it, read it back - the Kaggle flow
data = pd.DataFrame({
    'tenure_months':   tenure_months,
    'monthly_charges': monthly_charges,
    'support_calls':   support_calls,
    'contract':        contract,
    'churned':         churned,
})
data.to_csv('customers.csv', index=False)
print('saved customers.csv')

df = pd.read_csv('customers.csv')
print(df.shape)
df.head()

Always **look before you build**. Three questions for *any* new dataset:

1. What columns exist, and of what type? → `df.info()`
2. What do the numbers look like? → `df.describe()`
3. **Is the target balanced?** → `value_counts()` on the column you want to predict

In [ ]:
df.info()                                # types + missing-value check
print(df.describe().round(1))            # quick stats for the numeric columns
print(df['churned'].value_counts())      # is the target balanced?

## 3. Features `X` and target `y`, then the split

- **`X`** = the columns the model is *allowed* to use (the **features**)
- **`y`** = the column it must **predict** (the **target**, the answer)

Golden rule #1: **the answer must never sneak into `X`**. That is why we `drop` the `churned` column from `X`.

Then `train_test_split` sets some rows aside as a **test set** — a mini-exam the model never sees during training. Two new settings today:

- `stratify=y` → keeps the stayed/churned **ratio the same** in train and test (important whenever classes are imbalanced)
- `random_state=42` → the same "random" split every run, so results are reproducible

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['churned'])   # features: everything except the answer
y = df['churned']                  # target: the answer

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('train:', X_train.shape, '| test:', X_test.shape)
print(f'churn rate in train: {y_train.mean():.2f} | in test: {y_test.mean():.2f}')

## 4. Preprocessing — three things models can't eat

Raw data usually can't go straight into a model. Three blockers, three fixes:

| Problem | Fix |
|---|---|
| **Missing values** (`NaN`) | **impute** = fill them in, usually with the column's *median* |
| **Text columns** (models only understand numbers) | **one-hot encode** = one 0/1 column per category |
| **Very different scales** (charges 20–120 vs calls 0–8) | **scale** = bring columns to a comparable range |

And the golden rule that trips everyone up:

> **Fit preprocessing on the TRAINING data only, then apply it to the test data.**
> If the test set's own medians and means leak into your preprocessing, your evaluation becomes a lie. This is called **data leakage**.

In [ ]:
from sklearn.impute import SimpleImputer

# real data always has holes - let's add a few ourselves
holes = rng.choice(df.index, size=10, replace=False)
df.loc[holes, 'monthly_charges'] = np.nan
print('missing monthly_charges:', df['monthly_charges'].isna().sum())

# refresh the split AFTER adding holes, so train and test both may contain some
X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# impute: learn the median from TRAIN, apply it to both sets
num_cols = ['tenure_months', 'monthly_charges', 'support_calls']
imputer = SimpleImputer(strategy='median')
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])   # fit + transform on train
X_test[num_cols]  = imputer.transform(X_test[num_cols])        # transform ONLY on test!

print('missing left in train:', X_train[num_cols].isna().sum().sum(), '| test:', X_test[num_cols].isna().sum().sum())

### One-hot encoding the text column

`contract` holds words — models need numbers. `pd.get_dummies` turns one text column into one 0/1 column per value (`contract_month-to-month`, `contract_one-year`, `contract_two-year`). A customer on a two-year contract then shows a `1` in that column and `0` in the others.

In [ ]:
X_train = pd.get_dummies(X_train, columns=['contract'])
X_test  = pd.get_dummies(X_test,  columns=['contract'])

X_train.head()
# (advanced note: scikit-learn's OneHotEncoder inside a Pipeline keeps train/test columns
#  aligned automatically - see section 9. get_dummies is fine while both sets contain
#  all the same categories, which is the case here.)

### Scaling — whose distances count?

`KNN` (coming up) measures **distances** between customers. But `monthly_charges` ranges from ~20 to ~120 while `support_calls` ranges from 0 to 8 — so charges would dominate every distance calculation, just because its units are bigger. `StandardScaler` fixes this by giving every column **mean 0 and standard deviation 1**.

Two things to know:

- Tree-based models (decision tree, random forest) **don't care** about scaling — they split one column at a time.
- Distance- and equation-based models (KNN, logistic regression) **do** benefit. Scaling everything is the safe habit.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # learn mean/std from TRAIN only
X_test_s  = scaler.transform(X_test)        # apply the SAME scaling to test

print('scaled means ~ 0:', X_train_s.mean(axis=0).round(3)[:4])
print('scaled stds  ~ 1:', X_train_s.std(axis=0).round(3)[:4])

## 5. Model building — five models, one loop

First, a **baseline**: a deliberately dumb model that always predicts the majority class ('stayed'). Every real model must beat it — otherwise it has learned nothing at all. Then four real classifiers. Notice they all share the same API you already know from Day 1: **`fit` → `predict`**.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    'Baseline (predict majority)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression':         LogisticRegression(max_iter=500),
    'KNN (k=5)':                   KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':               DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':               RandomForestClassifier(n_estimators=200, random_state=42),
}

for name, model in models.items():
    model.fit(X_train_s, y_train)      # scaled data is safe for all of these
    preds = model.predict(X_test_s)
    print(f'{name:30s} test accuracy: {accuracy_score(y_test, preds):.3f}')

### Accuracy lies — meet precision, recall and F1

Accuracy = the share of correct predictions. But imagine churn were rare: 90% of customers stay. A model that **never predicts churn** would score 90% accuracy — and be perfectly useless. So on top of accuracy we look at per-class measures:

- **Precision** (for churn): of the customers I *flagged* as churners, how many really churned? → high precision = few false alarms
- **Recall** (for churn): of all *real* churners, how many did I catch? → high recall = few missed customers
- **F1-score**: the balance of the two in one number

The **confusion matrix** shows every outcome: rows = the truth, columns = the prediction, diagonal = correct. `precision_recall` trade-off is a business decision: a retention campaign that costs money wants *precision*; a safety-critical alert wants *recall*.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

forest = models['Random Forest']    # the forest we already fitted in the loop
preds = forest.predict(X_test_s)

print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, target_names=['stayed', 'churned']))

### A quick trip to regression — predicting a *number*

Everything above is **classification** (predicting a category: churn or stay). **Regression** predicts a number: a house price, a temperature, a salary. The workflow is identical — only the model and the metrics change:

- **MAE** — on average, how far off is a prediction? (in the same unit as the target)
- **RMSE** — like MAE, but punishes big misses harder
- **R²** — how much better than 'just predict the average'? `1.0` = perfect, `0.0` = no better than the average, negative = worse

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# the 'y = 3x + noise' data from Day 1, now with 200 points
r = np.random.default_rng(0)
x_reg = r.uniform(0, 10, size=200).reshape(-1, 1)   # sklearn wants X 2-D: (rows, features)
y_reg = 3 * x_reg.ravel() + r.normal(0, 3, size=200)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    x_reg, y_reg, test_size=0.25, random_state=42
)

lin  = LinearRegression().fit(Xr_train, yr_train)
pred = lin.predict(Xr_test)

print('learned slope:', lin.coef_[0].round(2), '(the truth was 3)')
print('MAE :', round(mean_absolute_error(yr_test, pred), 2))
print('RMSE:', round(mean_squared_error(yr_test, pred) ** 0.5, 2))
print('R2  :', round(r2_score(yr_test, pred), 3))

## 6. Overfitting and underfitting — the #1 thing to know

- **Underfitting**: the model is too simple to catch the pattern → bad on train AND bad on test.
- **Overfitting**: the model **memorises** the training rows, quirks included → great on train, bad on test. Like a student who memorised last year's exam paper instead of learning the subject.

The **gap between the train score and the test score is your overfitting meter**. Watch it open up as we let a decision tree grow deeper (more depth = more memorising power; `None` = unlimited):

In [ ]:
for depth in [1, 2, 3, 5, 8, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)                     # trees don't need scaling - unscaled is fine
    tr = accuracy_score(y_train, tree.predict(X_train))
    te = accuracy_score(y_test,  tree.predict(X_test))
    print(f'max_depth = {str(depth):4s} | train: {tr:.3f} | test: {te:.3f} | gap: {tr - te:.3f}')

## 7. Cross-validation — one split is one opinion

Our test score depends on *which* rows happened to land in the test set — a lucky split gives a lucky score. **5-fold cross-validation** judges more fairly:

1. cut the **training data** into 5 slices ("folds")
2. train on 4 folds, check on the 5th — five times, each fold taking one turn as the checker
3. average the five scores

The test set stays locked away until the very end. *(You already peeked at the test set above — fine for learning, but in a serious project you compare models with cross-validation FIRST and touch the test set once, at the end.)*

In [ ]:
from sklearn.model_selection import cross_val_score

forest = RandomForestClassifier(n_estimators=200, random_state=42)
scores = cross_val_score(forest, X_train_s, y_train, cv=5, scoring='accuracy')

print('fold scores:', scores.round(3))
print(f'mean: {scores.mean():.3f}  (+/- {scores.std():.3f})')

## 8. Pipelines — make leakage impossible

So far we scaled by hand: fit on train, transform test… easy to forget, easy to get wrong. A **pipeline** chains preprocessing + model into **one object** that does it correctly by construction — and during cross-validation it even refits the scaler inside every fold, so nothing leaks:

`scaler → model` behaves as a single unit with the same `.fit()` / `.predict()` you already know.

In [ ]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

cv = cross_val_score(pipe, X_train, y_train, cv=5)   # pass the UNSCALED data - the pipeline scales it
print('CV accuracy:  ', round(cv.mean(), 3))

pipe.fit(X_train, y_train)
print('test accuracy:', round(pipe.score(X_test, y_test), 3))

## 9. Saving the trained model

A model worth keeping should be **saved** to a file, so anyone (or an app) can load and use it later without retraining. `joblib` is the standard tool.

In [ ]:
import joblib

joblib.dump(pipe, 'churn_knn_pipeline.joblib')
loaded = joblib.load('churn_knn_pipeline.joblib')

print('loaded model test accuracy:', round(loaded.score(X_test, y_test), 3))

## 10. Cheat sheet — the things to know

- **Look before you build**: `info()`, `describe()`, `value_counts()` on the target
- **Split first**, preprocess after — and fit preprocessing on **train only** (leakage!)
- **Always compare against a dumb baseline** — beating 'predict the majority' is the minimum bar
- **Accuracy lies on imbalanced data** → use precision / recall / F1 and the confusion matrix
- **The train↔test gap is the overfitting meter** → fix with a simpler model, fewer features, or more data
- **One split is one opinion** → cross-validate before trusting a number
- **Pipelines** bundle preprocessing + model so the honest path is the easy path
- **Save** your final model with `joblib`

Same loop, every time: **load → split → preprocess → fit → predict → evaluate (honestly).**

## Exercises

1. **Tune k**: loop `k` over `[1, 3, 5, 11, 21, 41]`, record the CV accuracy of a `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))` for each, and `plt.plot` accuracy against k. What happens at k=1 — and why?
2. **Feature engineering**: add a column `calls_per_month = support_calls / tenure_months` to `df` *before* the split. Does the CV accuracy improve?
3. **Read the report**: run `classification_report` for the decision tree AND the random forest. Which class is harder to predict, and how do the two models differ on it?
4. **Feel the split**: change `test_size` to 0.5, then to 0.1. What happens to the stability of the test score — and why?
5. **Full run-through**: take a Kaggle dataset (the Titanic dataset is perfect: <https://www.kaggle.com/competitions/titanic>) and run the entire loop end-to-end on it. Getting stuck on real data is the point — that is exactly what real ML work feels like.

Finished? You now know the complete loop — keep practicing it on real datasets.